In [ ]:
import ast
import json
import os
import re
import sqlite3
import time

import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from sklearn.preprocessing import StandardScaler

load_dotenv()

In [ ]:
# media_df = pd.read_csv("../data/media.csv")
user_df = pd.read_csv("../data/users.csv")

In [ ]:
user_df.columns

In [ ]:
user_df.shape

In [ ]:
user_df.iloc[0]

In [ ]:
d = json.loads(user_df.iloc[0]["statistics"].replace("'", '"'))
print(json.dumps(d, indent=4))

In [ ]:
user_df["statistics"] = user_df["statistics"].apply(lambda x: ast.literal_eval(x))

In [ ]:
# Extract minutes watched and scale
user_df["minutes_watched"] = user_df["statistics"].apply(
    lambda x: x["anime"].get("minutesWatched", 0)
)
user_df = user_df[user_df["minutes_watched"] != 0]
mins = user_df["minutes_watched"].to_numpy(dtype=np.float32)
mins = np.log1p(mins)
mins = StandardScaler().fit_transform(mins.reshape(-1, 1)).astype(np.float16)
user_df["minutes_watched"] = mins

In [ ]:
# Extract users global mean score and scale
mean_score = user_df["anime"]["meanScore"].to_numpy(dtype=np.float32)
mean_score = (
    StandardScaler().fit_transform(mean_score.reshape(-1, 1)).astype(np.float16)
)
user_df["meanScore"] = mean_score

In [ ]:
# Extract the anime ids of all anime with a meanScore >= 70
user_df["pos_ids"] = user_df["statistics"].apply(
    lambda x: {
        mid
        for s in x["anime"]["scores"]
        if s["meanScore"] >= 70
        for mid in s["mediaIds"]
    }
)

# Extract the anime ids of all anime with a meanScore < 40
user_df["pos_ids"] = user_df["statistics"].apply(
    lambda x: {
        mid
        for s in x["anime"]["scores"]
        if s["meanScore"] < 40
        for mid in s["mediaIds"]
    }
)

# Remove all users with no pos or neg ids
user_df = user_df[user_df["pos_ids"].apply(len) > 0]

In [ ]:
# Retrieve the embeddings for all pos_ids from sqlite db, average them and store them in a new column "history"
# Connect to the sqlite database
conn = sqlite3.connect("../data/anime.db")
cursor = conn.cursor()


# Retrieve embeddings for pos_ids and average them
def get_averaged_embedding(pos_ids):
    if not pos_ids:
        return np.zeros(384)  # Return zero vector if no pos_ids

    placeholders = ",".join("?" * len(pos_ids))
    query = f"SELECT embedding FROM embeddings WHERE id IN ({placeholders})"
    cursor.execute(query, tuple(pos_ids))

    embeddings = [np.frombuffer(row[0], dtype=np.float32) for row in cursor.fetchall()]
    return np.mean(embeddings, axis=0) if embeddings else np.zeros(384)


user_df["history"] = user_df["pos_ids"].apply(get_averaged_embedding)

conn.close()

- id --> dropped for training; used to identify
- username --> drop
- updatedAt --> drop
- stats:
  - minutesWatched --> log-transform and standard scale
  - meanScore (global meanScore) --> standard scale, use as sort of an optimism bias, i.e. does a user tend to give high or low scores
  - completion_rate --> COUNT(status=completed) / (COUNT(status=completed) + COUNT(status=dropped))
  - history --> average the embeddings of the completed anime with scores e.g. >=70

In [ ]:
np.savez_compressed("../data/user_dataset.npz", ids=ids, metadata=metadata)